# QAOA — Kvantna aproksimativna optimizacija

**Quantum Approximate Optimization Algorithm (QAOA)** je varijacioni kvantni algoritam koji aproksimira adijabatsku evoluciju koristeći gate-bazirane kvantne računare. Umesto kontinualne spore promene Hamiltonijana, QAOA diskretizuje evoluciju u $p$ naizmeničnih slojeva.

**Autori:** Farhi, Goldstone, Gutmann (2014)

## 1. Od adijabatske evolucije do QAOA

### Adijabatska evolucija — podsetnik

U adijabatskom kvantnom računarstvu, sistem evoluira pod Hamiltonijanom:

$$H(t) = \left(1 - \frac{t}{T}\right) H_0 + \frac{t}{T} H_C$$

gde je $H_0 = -\sum_i \hat{\sigma}_i^x$ mešač (mixer), a $H_C$ problemski (cost) Hamiltonijan. Ako je $T$ dovoljno veliko, sistem završava u osnovnom stanju $H_C$.

### Troterizacija

Trotterova aproksimacija (Suzuki–Trotter dekompozicija) diskretizuje vremensku evoluciju na $p$ koraka:

$$e^{-i H(t) \Delta t} \approx e^{-i H_C \gamma} \cdot e^{-i H_0 \beta}$$

Primenom ovog koraka $p$ puta dobijamo QAOA kolo dubine $p$:

$$|\psi_p(\boldsymbol{\gamma}, \boldsymbol{\beta})\rangle = \prod_{k=1}^{p} e^{-i \beta_k H_0}\, e^{-i \gamma_k H_C}\, |+\rangle^{\otimes n}$$

Parametri $\boldsymbol{\gamma} = (\gamma_1, \ldots, \gamma_p)$ i $\boldsymbol{\beta} = (\beta_1, \ldots, \beta_p)$ se klasično optimizuju.

### Veza sa AQC

| | AQC | QAOA |
|---|---|---|
| **Evolucija** | Kontinualna, $T \to \infty$ | Diskretna, $p$ slojeva |
| **Hardver** | D-Wave (analogni) | IBM, IonQ (gate-bazirani) |
| **Parametri** | Brzina žarenja | $\boldsymbol{\gamma}, \boldsymbol{\beta}$ |
| **Granica** | $p \to \infty$, QAOA → AQC | Konačno $p$ |
| **Garancija** | Adijabatska teorema | Aproksimativna |


## 2. Hamiltonijani u QAOA

### Cost Hamiltonijan $H_C$

Enkoduje problem koji rešavamo. Za QUBO problem $\min f(\mathbf{x})$, koristimo Izing Hamiltonijan identičan onom iz AQC:

$$H_C = \sum_i h_i \hat{\sigma}_i^z + \sum_{i<j} J_{ij} \hat{\sigma}_i^z \hat{\sigma}_j^z$$

Osnovno stanje $H_C$ odgovara optimalnom rešenju $\mathbf{x}^*$. Merenje u $\hat{\sigma}^z$ bazi daje kandidata za rešenje.

### Mixer Hamiltonijan $H_0$

Standardni mešač je transverzalno polje — isti kao u AQC:

$$H_0 = -\sum_{i=1}^{n} \hat{\sigma}_i^x$$

Uloga mešača je da **poveže** različite konfiguracije kako bi pretraga mogla da "skoči" između njih. Bez mešača, evolucija bi bila trivijalna.

### Unitarni operatori

Svaki sloj QAOA primenjuje dva unitarna operatora:

$$U_C(\gamma) = e^{-i \gamma H_C} = \prod_{i} e^{-i \gamma h_i \hat{\sigma}_i^z} \prod_{i<j} e^{-i \gamma J_{ij} \hat{\sigma}_i^z \hat{\sigma}_j^z}$$

$$U_0(\beta) = e^{-i \beta H_0} = \prod_{i=1}^{n} e^{i \beta \hat{\sigma}_i^x} = \prod_{i=1}^{n} R_x(2\beta)_i$$

gde je $R_x(\theta) = e^{-i\frac{\theta}{2}\hat{\sigma}^x}$ rotacija oko $x$-ose na Blohovoj sferi.

Svaki od ovih operatora se direktno implementira kao niz kvantnih kapija.

## 3. Struktura QAOA kola

### Inicijalizacija

Svi qubiti se postavljaju u ravnomernu superpoziciju primenom Adamarove kapije na svaki qubit:

$$|+\rangle^{\otimes n} = H^{\otimes n} |0\rangle^{\otimes n} = \frac{1}{\sqrt{2^n}} \sum_{\mathbf{x} \in \{0,1\}^n} |\mathbf{x}\rangle$$

Ovo je osnovno stanje mešača $H_0$ i idealno polazno stanje jer ne favorizuje nijedno rešenje.

### Jedan sloj ($p = 1$)

```
|0⟩ ─── H ─── U_C(γ₁) ─── U_0(β₁) ─── meri
|0⟩ ─── H ─── U_C(γ₁) ─── U_0(β₁) ─── meri
|0⟩ ─── H ─── U_C(γ₁) ─── U_0(β₁) ─── meri
```

### $p$ slojeva

```
|0⟩⊗ⁿ ── H⊗ⁿ ── [U_C(γ₁) U_0(β₁)] ── [U_C(γ₂) U_0(β₂)] ── ⋯ ── [U_C(γₚ) U_0(βₚ)] ── meri
         └─────── inicijalizacija ──┘   └───────── p slojeva ─────────────────────────┘
```

Ukupno $2p$ parametara: $\boldsymbol{\gamma} \in [0, 2\pi)^p$, $\boldsymbol{\beta} \in [0, \pi)^p$.

### Implementacija $U_C(\gamma)$ kao kapije

Za svaku granu $(i, j)$ u grafu problema:

$$e^{-i \gamma J_{ij} \hat{\sigma}_i^z \hat{\sigma}_j^z} \;=\; \text{CNOT}_{ij} \cdot R_z(2\gamma J_{ij})_j \cdot \text{CNOT}_{ij}$$

Za linearni član na qubitu $i$:

$$e^{-i \gamma h_i \hat{\sigma}_i^z} = R_z(2\gamma h_i)_i$$

Za standardni mešač, svaki qubit dobija $R_x$ rotaciju:

$$e^{i \beta \hat{\sigma}_i^x} = R_x(-2\beta)_i$$

## 4. Varijaciona optimizacija

QAOA je varijacioni kvantni algoritam (VQA). Cilj je minimizovati **očekivanu vrednost** cost Hamiltonijana:

$$F_p(\boldsymbol{\gamma}, \boldsymbol{\beta}) = \langle \psi_p | H_C | \psi_p \rangle$$

### Tok optimizacije

```
  Inicijalni parametri (γ, β)
           ↓
  Priprema |ψₚ(γ, β)⟩ na kvantnom računaru
           ↓
  Merenje ⟨H_C⟩ (više puta za statistiku)
           ↓
  Klasični optimizer ažurira (γ, β)
           ↓
  Ponavljaj dok nije konvergencija
           ↓
  Finalno merenje → kandidat za rešenje
```

### Klasični optimizatori

| Optimizer | Tip | Prednost |
|---|---|---|
| **COBYLA** | Bez gradijenta | Robustan na šum |
| **SPSA** | Stohastički gradijent | Efikasan za šumne evaluacije |
| **L-BFGS-B** | Gradijent | Brz za glatke probleme |
| **Nelder-Mead** | Bez gradijenta | Jednostavan, spor |

### Problem ravnih platoa (Barren Plateaus)

Za duboka kola ($p$ veliko, $n$ veliko), gradijent funkcije cilja eksponencijalno opada:

$$\text{Var}\left[\frac{\partial F_p}{\partial \gamma_k}\right] \in O\left(\frac{1}{2^n}\right)$$

Ovo je fundamentalni problem varijacionih kvantnih algoritama — gradijent postaje numerički nula i optimizacija ne napreduje. Aktivna oblast istraživanja.

## 5. Aproksimativni odnos i dubina $p$

### Aproksimativni odnos

Za problem maksimizacije $\max f(\mathbf{x})$ sa optimalnom vrednosti $f^*$, QAOA daje vrednost $F_p$. **Aproksimativni odnos** je:

$$\alpha_p = \frac{F_p}{f^*} \in [0, 1]$$

$\alpha_p = 1$ znači optimalno rešenje; veće $p$ daje bolji $\alpha_p$.

### Garantovane granice

Za **Max-Cut** na 3-regularnim grafovima, Farhi et al. dokazali su:

$$\alpha_1 \geq \frac{3}{4} = 0.75 \quad (p=1)$$

Poređenje sa klasičnim algoritmima:

| Algoritam | Aproksimativni odnos za Max-Cut |
|---|---|
| Nasumičan rez | $1/2 = 0.5$ |
| **QAOA $p=1$** | $\geq 0.7498...$ |
| Goemans–Williamson (SDP) | $\approx 0.8786$ |
| Optimum | $1.0$ (NP-težak) |

### Konvergencija ka AQC za $p \to \infty$

**Teorema (Farhi et al., 2014):** Za svaki $\epsilon > 0$, postoji $p^*$ takvo da QAOA sa $p \geq p^*$ slojeva daje rešenje unutar $\epsilon$ od optimalnog. U granici $p \to \infty$, QAOA postaje tačna simulacija adijabatske evolucije.

Praktično: za realne probleme, dubine $p \in [1, 10]$ su izvodljive na trenutnom hardveru.

## 6. Primer: Max-Cut na trouglu

### Problem

Graf $G = (V, E)$ sa $V = \{0, 1, 2\}$, $E = \{(0,1), (1,2), (0,2)\}$ (trougao). Cilj: maksimizovati broj grana između particija.

### Cost Hamiltonijan

Max-Cut se formuliše kao minimizacija (negiramo za minimum):

$$H_C = -\frac{1}{2}\sum_{(i,j)\in E}(1 - \hat{\sigma}_i^z \hat{\sigma}_j^z) = \frac{1}{2}\sum_{(i,j)\in E} \hat{\sigma}_i^z \hat{\sigma}_j^z - \frac{|E|}{2}$$

Za naš trougao (ignorišući konstantu):

$$H_C \propto \hat{\sigma}_0^z\hat{\sigma}_1^z + \hat{\sigma}_1^z\hat{\sigma}_2^z + \hat{\sigma}_0^z\hat{\sigma}_2^z$$

### Optimalna rešenja

Trougao je **neparan ciklus** — nije bipartitan, pa nije moguće preseći sve 3 grane. Maksimalni rez je 2 grane:

| Particija | Presečene grane |
|---|---|
| $\{0\} \mid \{1,2\}$ | $(0,1), (0,2)$ = **2** |
| $\{1\} \mid \{0,2\}$ | $(0,1), (1,2)$ = **2** |
| $\{2\} \mid \{0,1\}$ | $(1,2), (0,2)$ = **2** |

Sva tri rešenja su podjednako optimalna.

### QAOA kolo za $p=1$

```
q0: ─ H ─ Rz(2γJ₀₁) ─ CNOT ─ Rz(2γJ₀₂) ─ CNOT ─ Rx(2β) ─ meri
               │       ctrl─target          ctrl─target
q1: ─ H ─────────────── ZZ ────────────── ZZ ──────── Rx(2β) ─ meri  
q2: ─ H ─────────────────────────────────────────────── Rx(2β) ─ meri
```

Sa optimalnim parametrima $\gamma^*, \beta^*$, merenje daje jedno od 3 optimalna rešenja sa visokom verovatnoćom.

## 7. QAOA vs. klasični algoritmi

### Potencijalna kvantna prednost

Za QAOA sa $p$ slojeva na $n$ qubita:
- Kolo ima dubinu $O(p)$ i zahteva $O(p \cdot |E|)$ kapija
- Broj parametara: $2p$
- Svako merenje daje kandidata za rešenje u $O(1)$

Klasični algoritmi rade u polinomijalnom vremenu za neke aproksimacije, ali QAOA može biti bolje u praksi za specifične instance — ovo je otvoreno pitanje istraživanja.

### Ograničenja trenutnog hardvera (NISQ era)

| Ograničenje | Efekat na QAOA |
|---|---|
| Šum kapija | Greške se akumuliraju sa dubinom $p$ |
| Kratko vreme koherencije | Ograničava maksimalno $p$ |
| Ograničena konektivnost | Dodatne SWAP kapije povećavaju dubinu |
| Barren plateaus | Teška optimizacija za veliki $n$ |

Trenutno, QAOA je izvodljiv za $n \lesssim 100$ qubita i $p \lesssim 5$ na realnom hardveru.

### Varijante i proširenja

| Varijanta | Opis |
|---|---|
| **QAOA+** | Poboljšani mešač koji poštuje ograničenja |
| **ma-QAOA** | Mešač prilagođen problemu (mixer-adapted) |
| **Recursive QAOA** | Rekurzivno smanjenje broja qubita |
| **ADAPT-QAOA** | Adaptivni izbor kapija po slojevima |
| **FALQON** | Bez varijacione optimizacije (feedback) |

## Rezime

| Pojam | Suština |
|---|---|
| **QAOA** | Varijacioni algoritam — gate-bazirana aproksimacija AQC |
| **$p$ slojeva** | Dubina kola; $p \to \infty$ konvergira ka AQC |
| **$H_C$** | Cost Hamiltonijan — enkoduje problem (isti kao Izing u AQC) |
| **$H_0$** | Mešač — transverzalno polje $-\sum \hat\sigma^x$ |
| **$U_C(\gamma)$** | Fazni operator — primenjuje cost |
| **$U_0(\beta)$** | Mešač operator — $R_x$ rotacije |
| **$(\boldsymbol{\gamma}, \boldsymbol{\beta})$** | $2p$ parametara, optimizuju se klasično |
| **Barren plateaus** | Eksponencijalno mali gradijenti za duboka kola |
| **Aproks. odnos** | $\alpha_p = F_p / f^* \to 1$ kako $p \to \infty$ |

**Ograničenja:** QAOA ne garantuje eksponencijalno ubrzanje nad klasičnim algoritmima — kvantna prednost ostaje otvoreno teorijsko pitanje. Na NISQ uređajima, šum ograničava praktičnu dubinu.

> **Veza sa prethodnim:** AQC (D-Wave) koristi analognu kontinualnu evoluciju; QAOA je njena diskretna gate-bazirana aproksimacija pogodna za IBM, IonQ i slične računare.